# 07 — One frozen-test release gate

**Objectives**

- Evaluate the exact selected Logged Model on the later-time test partition.
- Combine classic MLflow classifier diagnostics with an explicit business gate.
- Make one `adopt`, `reject`, or `inconclusive` decision.

**Prerequisite:** lesson 06 fixed the candidate and threshold.

This function is idempotent for the same selected artifact and policy: rerunning
returns the existing decision rather than repeatedly opening the test. If code,
dependencies, model, or policy differ after this dataset version is consumed,
the gate refuses to reuse it. Create a new frozen-test version before making a
new release claim.


In [ ]:
import pandas as pd

from aai_local_classification.learning import short_digest, state_exists
from aai_local_classification.workflow import (
    run_candidate_selection,
    run_frozen_test_gate,
)
from aai_local_classification.learning import study_root
from aai_local_classification.settings import load_settings

settings = load_settings()
root = study_root()
print(f"Course state: {root}")
print(f"Experiment: {settings.experiment_name}")


In [ ]:
if not state_exists("selection.json"):
    selection = run_candidate_selection(settings, root)
else:
    selection = None
decision = run_frozen_test_gate(settings, root, selection)
pd.Series(
    {
        "decision": decision.decision.value,
        "selected_candidate": decision.selected_candidate,
        "selected_run_id": decision.selected_run_id,
        "test_run_id": decision.test_run_id,
        "threshold": decision.threshold,
        "dataset": short_digest(decision.dataset_sha256),
    }
).to_frame("value")


In [ ]:
pd.DataFrame(
    [
        {"check": name, "passed": passed}
        for name, passed in decision.checks.model_dump().items()
    ]
)


In [ ]:
important = [
    "test_average_precision",
    "test_roc_auc",
    "test_precision",
    "test_recall",
    "test_f1",
    "test_brier_score",
    "test_cost_per_1000",
    "test_maximum_slice_recall_gap",
]
pd.Series({name: decision.metrics[name] for name in important}).to_frame("test value")


The native MLflow classic evaluator logs standard classifier diagnostics in the
result run. The explicit gate remains authoritative because it uses the
validation-selected business threshold, declared costs, and operational slices.
This is intentionally not MLflow GenAI evaluation.

### Exercise

Which result should be `inconclusive` rather than `reject`? Give one example
involving missing evidence and one involving statistical uncertainty.

**Hint:** an invalid test extract or a confidence interval spanning the minimum
effect cannot establish that the model is bad—or good.

**Checkpoint:** the exact selected artifact either passed every declared check
or remains unpromoted. The decision links the dataset digest, model/run IDs,
threshold, metrics, and test run.

Next: **08_registry_and_inference.ipynb**.
